In [1]:
# === 00_preprocess_data.ipynb — Cellule 1 : Imports & paramètres globaux ===

# ── Imports standards ──────────────────────────────────────────────────────────
import os, gc, json, logging, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

# ── (Optionnel) TensorFlow uniquement pour seeding / tf.data démo en fin ───────
try:
    import tensorflow as tf  # pas requis pour la prépa des tensors, uniquement utile pour la démo tf.data
    TF_AVAILABLE = True
except Exception:
    TF_AVAILABLE = False

# ── Affichage propre ───────────────────────────────────────────────────────────
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

# ── Paramètres principaux (rejouables) ────────────────────────────────────────
SEED          = 42            # reproductibilité (NumPy + TF si dispo)
MAX_LEN       = 206           # longueur cible (padding/truncation)
VOCAB         = {"A":1,"C":2,"G":3,"U":4}  # 0 = PAD / inconnu
PAD_ID        = 0

# BPP: 'band' pour une bande locale compacte, 'full' pour matrice dense 206x206
BPP_MODE      = "band"        # "band" (recommandé V1) ou "full"
BAND          = 20            # demi-largeur de bande (±20) si mode "band"
LOCAL_W       = 2*BAND + 1    # largeur locale (41 si BAND=20)

# Stratégie de masque par position à livrer dans le NPZ
#  - "strict": position valide si DMS ET 2A3 présents (recommandé)
#  - "mean"  : moyenne des deux canaux -> {0, 0.5, 1}
#  - "max"   : valide si ≥1 canal présent
MASK_STRATEGY = "strict"

# ── Chemins d'entrée/sortie (adapter si besoin) ───────────────────────────────
DATA_DIR   = Path(".")
BPP_ROOT   = Path("Ribonanza_bpp_files/extra_data")

TRAIN_CSV  = DATA_DIR / "train_data.csv"   # ou "train_data.csv" si dispo
TEST_CSV   = DATA_DIR / "test_sequences.csv"           # optionnel (pas utilisé en V1)

# Nom du cache à produire (inclut le mode BPP)
CACHE_DIR  = Path("./cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_NPZ    = CACHE_DIR / f"dataset_npz_{BPP_MODE}.npz"
MANIFEST   = CACHE_DIR / f"dataset_manifest_{BPP_MODE}.json"
USE_SN_FILTER = True     # si True, on garde les lignes SN_filter==1
SUBSET_N      = 20000     # ex. mets 20000 pour tester sur un sous-ensemble
# ── Seeding reproductible ─────────────────────────────────────────────────────
np.random.seed(SEED)
if TF_AVAILABLE:
    try:
        tf.random.set_seed(SEED)
    except Exception:
        pass

# ── Logs lisibles ────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
print("== Prépa V1 — paramètres ==")
print({
    "SEED": SEED,
    "MAX_LEN": MAX_LEN,
    "VOCAB": VOCAB,
    "BPP_MODE": BPP_MODE,
    "BAND": BAND,
    "LOCAL_W": LOCAL_W,
    "MASK_STRATEGY": MASK_STRATEGY,
    "TRAIN_CSV": str(TRAIN_CSV),
    "BPP_ROOT": str(BPP_ROOT),
    "OUT_NPZ": str(OUT_NPZ),
})
print("TensorFlow dispo ?", TF_AVAILABLE)


2025-10-23 15:23:51.307247: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


== Prépa V1 — paramètres ==
{'SEED': 42, 'MAX_LEN': 206, 'VOCAB': {'A': 1, 'C': 2, 'G': 3, 'U': 4}, 'BPP_MODE': 'band', 'BAND': 20, 'LOCAL_W': 41, 'MASK_STRATEGY': 'strict', 'TRAIN_CSV': 'train_data.csv', 'BPP_ROOT': 'Ribonanza_bpp_files/extra_data', 'OUT_NPZ': 'cache/dataset_npz_band.npz'}
TensorFlow dispo ? True


In [2]:
assert TRAIN_CSV.exists(), f"Introuvable: {TRAIN_CSV}"

# On lit d'abord juste l'entête pour détecter les colonnes de réactivité
head = pd.read_csv(TRAIN_CSV, nrows=1, low_memory=False)
react_cols = [c for c in head.columns if c.startswith("reactivity_") and not c.startswith("reactivity_error_")]
err_cols   = [c for c in head.columns if c.startswith("reactivity_error_")]

# Colonnes minimales utiles (on peut en ajouter si besoin)
base_cols = ["sequence_id", "sequence", "experiment_type", "SN_filter"]
usecols = base_cols + react_cols + err_cols

# Lecture du gros CSV avec uniquement les colonnes nécessaires
df = pd.read_csv(TRAIN_CSV, usecols=usecols, low_memory=False)

# Filtre qualité optionnel
if "SN_filter" in df.columns and USE_SN_FILTER:
    before = len(df)
    df = df[df["SN_filter"] == 1].copy()
    print(f"[SN_filter] {before} → {len(df)} lignes (conservé SN_filter==1)")

# Sous-échantillon optionnel pour tests rapides
if SUBSET_N is not None and len(df) > SUBSET_N:
    df = df.sample(n=SUBSET_N, random_state=SEED).reset_index(drop=True)
    print(f"[SUBSET] échantillon aléatoire de {SUBSET_N} lignes")

# Sanity checks
df["seq_len"] = df["sequence"].astype(str).str.len()
print({
    "rows": len(df),
    "react_cols": len(react_cols),
    "err_cols": len(err_cols),
    "len_min": int(df["seq_len"].min()),
    "len_med": float(df["seq_len"].median()),
    "len_max": int(df["seq_len"].max()),
    "exp_counts": df["experiment_type"].value_counts(dropna=False).to_dict()
})


[SN_filter] 1643680 → 438239 lignes (conservé SN_filter==1)
[SUBSET] échantillon aléatoire de 20000 lignes
{'rows': 20000, 'react_cols': 206, 'err_cols': 206, 'len_min': 115, 'len_med': 177.0, 'len_max': 206, 'exp_counts': {'DMS_MaP': 10357, '2A3_MaP': 9643}}


In [3]:
# === Cellule 3 (REMPLACEMENT) : Y + clipping + masques NaN par canal, avec dédup/agg ===
# On a des doublons (sequence_id, experiment_type) dans train_data.csv ⇒ on déduplique avant unstack.
# Deux stratégies possibles :
#   - "best" : on garde la meilleure ligne (SN_filter=1 en priorité, puis plus haut signal_to_noise, puis plus de reads)
#   - "mean" : on moyenne les colonnes de réactivité au sein de chaque groupe
AGG_STRATEGY = "best"   # change en "mean" si tu préfères moyenner

key = ["sequence_id", "experiment_type"]

if AGG_STRATEGY == "best":
    sort_cols, sort_asc = [], []
    if "SN_filter" in df.columns:          sort_cols.append("SN_filter");        sort_asc.append(False)
    if "signal_to_noise" in df.columns:    sort_cols.append("signal_to_noise");  sort_asc.append(False)
    if "reads" in df.columns:              sort_cols.append("reads");            sort_asc.append(False)

    df_sorted = df.sort_values(sort_cols, ascending=sort_asc) if sort_cols else df
    df_dedup  = df_sorted.drop_duplicates(subset=key, keep="first").copy()

elif AGG_STRATEGY == "mean":
    agg_dict = {"sequence": "first"}  # on garde une séquence arbitraire du groupe
    for c in react_cols:
        agg_dict[c] = "mean"
    for c in [c for c in df.columns if c.startswith("reactivity_error_")]:
        agg_dict[c] = "mean"
    if "SN_filter" in df.columns:       agg_dict["SN_filter"] = "max"
    if "signal_to_noise" in df.columns: agg_dict["signal_to_noise"] = "max"
    if "reads" in df.columns:           agg_dict["reads"] = "max"
    df_dedup = df.groupby(key, as_index=False).agg(agg_dict)

else:
    raise ValueError("AGG_STRATEGY doit être 'best' ou 'mean'")

# Alerte si des sequence_id ont plusieurs séquences (rare) – on garde la première.
conflicts = df_dedup.groupby("sequence_id")["sequence"].nunique().gt(1)
if conflicts.any():
    n_conflict = int(conflicts.sum())
    print(f"[WARN] {n_conflict} sequence_id possèdent plusieurs 'sequence' différentes (on garde 'first').")

# 1) Pivot propre (1 ligne par sequence_id, 2 colonnes par expérience DMS/2A3)
keep = ["sequence_id", "sequence", "experiment_type"] + react_cols
wide = (df_dedup[keep]
        .set_index(["sequence_id", "experiment_type"])
        .sort_index()
        .unstack("experiment_type"))

# 2) Construire Y = (N, L, 2) et M = (N, L, 2) (1 si observé)
def build_Y_and_mask(wide, max_len=MAX_LEN):
    Ys, Ms = [], []

    def _get(wide, colname, exp_name):
        # Récupère la colonne (colname, exp_name) si elle existe, sinon tableau de NaN
        try:
            return wide[(colname, exp_name)].to_numpy(dtype=float)
        except KeyError:
            return np.full((len(wide),), np.nan, dtype=float)

    for k in range(1, max_len+1):
        col = f"reactivity_{k:04d}"

        dm  = _get(wide, col, "DMS_MaP")
        a3  = _get(wide, col, "2A3_MaP")

        m_dm = (~np.isnan(dm)).astype(np.float32)
        m_a3 = (~np.isnan(a3)).astype(np.float32)

        dm  = np.nan_to_num(dm, nan=0.0)
        a3  = np.nan_to_num(a3, nan=0.0)

        Ys.append(np.stack([dm, a3], axis=-1))
        Ms.append(np.stack([m_dm, m_a3], axis=-1))

    Y = np.stack(Ys, axis=1).astype(np.float32)         # (N, L, 2)
    Y = np.clip(Y, 0.0, 1.0)                            # clipping officiel [0,1]
    M = np.stack(Ms, axis=1).astype(np.float32)         # (N, L, 2)
    return Y, M

Y, M_nan = build_Y_and_mask(wide, MAX_LEN)
print("Y:", Y.shape, Y.dtype, "M_nan:", M_nan.shape, M_nan.dtype)


Y: (19357, 206, 2) float32 M_nan: (19357, 206, 2) float32


In [4]:
# === Cellule 4 : Encodage séquences + padding/troncature ===
VOCAB = dict(VOCAB)  # copie locale

# Une seule séquence par sequence_id (on prend DMS si dispo sinon 2A3)
seq_dms = wide[("sequence","DMS_MaP")]
seq_2a3 = wide[("sequence","2A3_MaP")]
seq_str = seq_dms.where(seq_dms.notna(), seq_2a3)

valid_idx = seq_str.notna().to_numpy()
seq_ids   = np.array(wide.index.tolist())[valid_idx]
seq_str   = seq_str[valid_idx].astype(str).to_numpy()
Y         = Y[valid_idx]
M_nan     = M_nan[valid_idx]

def clean_seq(s: str) -> str:
    s = s.upper().replace("T","U")
    return "".join(ch if ch in VOCAB else "N" for ch in s)

def encode_and_pad(s: str, max_len=MAX_LEN, pad_id=PAD_ID) -> np.ndarray:
    s  = clean_seq(s)
    ids = [VOCAB.get(ch, pad_id) for ch in s]
    if len(ids) < max_len:
        ids += [pad_id] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return np.array(ids, dtype=np.int16)

X_seq = np.stack([encode_and_pad(s) for s in seq_str])
X_len = np.array([len(s) for s in seq_str], dtype=np.int16)  # longueur “réelle” (avant pad/tronc)

print("X_seq:", X_seq.shape, X_seq.dtype, "| X_len min/med/max:", X_len.min(), np.median(X_len), X_len.max())


X_seq: (19357, 206) int16 | X_len min/med/max: 115 177.0 206


In [5]:
# === Cellule 5 : Masque final par position W_pos (N, L) ===

# 1) masque “NaN” réduit à 1 canal par position (strict/mean/max)
if MASK_STRATEGY == "strict":
    W_from_nan = M_nan.min(axis=-1)          # 1 si DMS ET 2A3 dispos
elif MASK_STRATEGY == "mean":
    W_from_nan = M_nan.mean(axis=-1)         # {0, 0.5, 1}
elif MASK_STRATEGY == "max":
    W_from_nan = M_nan.max(axis=-1)          # 1 si au moins 1 canal dispo
else:
    raise ValueError(f"MASK_STRATEGY inconnu: {MASK_STRATEGY}")

# 2) masque de longueur: 1 si position < longueur réelle, 0 sinon
pos_idx = np.arange(MAX_LEN)[None, :]          # (1, L)
len_mask = (pos_idx < X_len[:, None]).astype(np.float32)  # (N, L)

# 3) masque final: intersection logique (on multiplie)
W_pos = (W_from_nan * len_mask).astype(np.float32)

# 4) filtre de sécurité: on enlève les séquences sans aucune position valide
keep_row = W_pos.sum(axis=1) > 0
X_seq  = X_seq[keep_row]
X_len  = X_len[keep_row]
Y      = Y[keep_row]
W_pos  = W_pos[keep_row]
seq_ids= seq_ids[keep_row]

print("W_pos:", W_pos.shape, W_pos.dtype, " | mean:", float(W_pos.mean()))
print("Séquences gardées:", int(keep_row.sum()))


W_pos: (494, 206) float32  | mean: 0.45427656173706055
Séquences gardées: 494


In [6]:
# === Cellule 6 : Ingestion BPPM (band) ===
from pathlib import Path
import pandas as pd

def index_bpp_folder(root: Path) -> dict[str, Path]:
    root = Path(root)
    return {p.stem: p for p in root.rglob("*.txt")}

def load_band_from_txt_fast(path: Path, band=BAND, Lmax=MAX_LEN) -> np.ndarray:
    # lit (i j p) → conserve uniquement |i-j|<=band, normalise par max local
    df = pd.read_csv(
        path, sep=r"\s+", header=None, names=["i","j","p"],
        engine="c", dtype={"i":np.int32, "j":np.int32, "p":np.float32}
    )
    if df.empty:
        return np.zeros((Lmax, 2*band+1), dtype=np.float32)

    df["i"] -= 1; df["j"] -= 1
    df = df[(df["i"]>=0)&(df["i"]<Lmax)&(df["j"]>=0)&(df["j"]<Lmax)]
    m = float(df["p"].max())
    if m > 0:
        df["p"] /= m

    d = (df["j"] - df["i"]).to_numpy()
    keep = np.abs(d) <= band
    if not np.any(keep):
        return np.zeros((Lmax, 2*band+1), dtype=np.float32)

    i = df["i"].to_numpy()[keep]
    j = df["j"].to_numpy()[keep]
    p = df["p"].to_numpy()[keep]
    b = j - i + band  # colonne de la bande

    band_mat = np.zeros((Lmax, 2*band+1), dtype=np.float32)
    np.maximum.at(band_mat, (i, b), p)
    b_rev = i - j + band
    np.maximum.at(band_mat, (j, b_rev), p)
    return band_mat

# index des fichiers BPP
bpp_paths = index_bpp_folder(BPP_ROOT)
print("Fichiers BPP indexés:", len(bpp_paths))

# construction alignée sur seq_ids gardés
BPPM_band = np.zeros((len(seq_ids), MAX_LEN, LOCAL_W), dtype=np.float32)
missing = 0
for n, sid in enumerate(seq_ids):
    p = bpp_paths.get(str(sid))
    if p is None:
        missing += 1
        continue
    BPPM_band[n] = load_band_from_txt_fast(p, band=BAND, Lmax=MAX_LEN)

print("BPPM_band:", BPPM_band.shape, "| missing:", missing)


Fichiers BPP indexés: 2112720
BPPM_band: (494, 206, 41) | missing: 0


In [7]:
with open("train_data.csv", "r") as f:
    n_lines = sum(1 for _ in f)
print(f"Nombre de lignes : {n_lines}")


Nombre de lignes : 1643681


In [8]:
# === Cellule 7 : Sauvegarde NPZ + manifeste JSON ===
CACHE_DIR.mkdir(parents=True, exist_ok=True)

to_save = {
    "seq_ids":  seq_ids,
    "X_seq":    X_seq.astype(np.int32),
    "X_len":    X_len.astype(np.int16),
    "Y":        Y.astype(np.float32),
    "W_pos":    W_pos.astype(np.float32),
}
if BPP_MODE == "band":
    to_save["BPPM_band"] = BPPM_band.astype(np.float32)

np.savez_compressed(OUT_NPZ, **to_save)

manifest = {
    "seed": SEED,
    "max_len": MAX_LEN,
    "vocab": VOCAB,
    "pad_id": PAD_ID,
    "mask_strategy": MASK_STRATEGY,
    "bpp_mode": BPP_MODE,
    "band": BAND,
    "local_w": LOCAL_W,
    "train_csv": str(TRAIN_CSV),
    "bpp_root": str(BPP_ROOT),
    "npz_path": str(OUT_NPZ),
    "shapes": {k: tuple(v.shape) if hasattr(v, "shape") else None for k, v in to_save.items()},
}
with open(MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)

print("Écrit:", OUT_NPZ)
print("Manifeste:", MANIFEST)
print("Shapes:", manifest["shapes"])


Écrit: cache/dataset_npz_band.npz
Manifeste: cache/dataset_manifest_band.json
Shapes: {'seq_ids': (494,), 'X_seq': (494, 206), 'X_len': (494,), 'Y': (494, 206, 2), 'W_pos': (494, 206), 'BPPM_band': (494, 206, 41)}


In [9]:
# === Cellule 8b (fix) : Split stratifié (longueur × couverture) ===
rng = np.random.default_rng(SEED)
N = len(X_seq)

lens = (X_seq != PAD_ID).sum(axis=1)
cov  = W_pos.mean(axis=1)

# Bins -> Series (pas ndarrays) pour pouvoir concaténer proprement
len_bins_s = pd.Series(pd.cut(lens, bins=[0,170,190,1000],
                              labels=["<=170","171-190",">190"], include_lowest=True))
cov_bins_s = pd.Series(pd.cut(cov,  bins=[-1e9, 0.33, 0.66, 1.0],
                              labels=["lowcov","midcov","highcov"], include_lowest=True))

# Sécurité si des NaN apparaissent (rare, mais au cas où)
len_bins_s = len_bins_s.astype(str).fillna("lenNA")
cov_bins_s = cov_bins_s.astype(str).fillna("covNA")

# Étiquette de strate
strata = len_bins_s.str.cat(cov_bins_s, sep="|").to_numpy()

# Répartition par strate
from collections import defaultdict
by_stratum = defaultdict(list)
for i, s in enumerate(strata):
    by_stratum[s].append(i)

train_idx, valid_idx = [], []
for s, ids in by_stratum.items():
    ids = np.array(ids)
    rng.shuffle(ids)
    k = int(0.8 * len(ids))
    if k == 0 and len(ids) > 1:
        k = 1
    train_idx.append(ids[:k])
    valid_idx.append(ids[k:])
train_idx = np.concatenate(train_idx) if train_idx else np.empty(0, dtype=int)
valid_idx = np.concatenate(valid_idx) if valid_idx else np.empty(0, dtype=int)

def _counts(labels):
    vc = pd.Series(labels).value_counts().sort_index()
    return (vc / vc.sum()).round(3).to_dict()

print("Strates total:", _counts(strata))
print("Strates train:", _counts(strata[train_idx]))
print("Strates valid:", _counts(strata[valid_idx]))
print(f"Split -> train={len(train_idx)} | valid={len(valid_idx)}")

def make_ds(x_seq, y, w_pos, bppm=None, batch=64, shuffle=False):
    w2 = np.repeat(w_pos[..., None], 2, axis=-1)
    if bppm is None:
        ds = tf.data.Dataset.from_tensor_slices((x_seq, y, w2))
    else:
        ds = tf.data.Dataset.from_tensor_slices(({"seq": x_seq, "bppm": bppm}, y, w2))
    if shuffle: ds = ds.shuffle(len(x_seq), seed=SEED)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

ds_train = make_ds(X_seq[train_idx], Y[train_idx], W_pos[train_idx],
                   bppm=BPPM_band[train_idx] if 'BPPM_band' in globals() and BPPM_band is not None else None,
                   shuffle=True)
ds_valid = make_ds(X_seq[valid_idx], Y[valid_idx], W_pos[valid_idx],
                   bppm=BPPM_band[valid_idx] if 'BPPM_band' in globals() and BPPM_band is not None else None,
                   shuffle=False)

# (Optionnel) persister le split
SPLIT_JSON = CACHE_DIR / f"split_indices_{BPP_MODE}.json"
with open(SPLIT_JSON, "w") as f:
    json.dump({"train_idx": train_idx.tolist(), "valid_idx": valid_idx.tolist()}, f)
print("Indices de split écrits dans:", SPLIT_JSON)


Strates total: {'171-190|midcov': 0.63, '<=170|midcov': 0.354, '>190|highcov': 0.016}
Strates train: {'171-190|midcov': 0.629, '<=170|midcov': 0.355, '>190|highcov': 0.015}
Strates valid: {'171-190|midcov': 0.63, '<=170|midcov': 0.35, '>190|highcov': 0.02}
Split -> train=394 | valid=100
Indices de split écrits dans: cache/split_indices_band.json


W0000 00:00:1761225961.914342   21785 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [10]:
# === Cellule 9 (remplacement) : Helpers (loss/metric + attention mask) ===
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def seq_padding_mask(x_int, pad_id=0):
    valid = tf.not_equal(x_int, tf.cast(pad_id, x_int.dtype))        # (B,L)
    attn  = tf.cast(valid[:, :, None] & valid[:, None, :], tf.bool)  # (B,L,L)
    return valid, attn

# On garde une loss "élémentaire" (Keras appliquera sample_weight)
def mae_clipped_elementwise(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
    return tf.abs(y_true - y_pred)  # (B,L,2)

# Mais on ajoute UNE métrique qui reproduit EXACTEMENT ton calcul "global"
class MaskedMAE(keras.metrics.Metric):
    def __init__(self, name="masked_mae", **kwargs):
        super().__init__(name=name, **kwargs)
        self.total_err = self.add_weight(name="total_err", initializer="zeros")
        self.total_w   = self.add_weight(name="total_w",   initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
        err = tf.abs(y_true - y_pred)                # (B,L,2)
        if sample_weight is not None:
            sw = tf.cast(sample_weight, err.dtype)   # (B,L,2)
            err = err * sw
            w   = tf.reduce_sum(sw)
        else:
            w   = tf.cast(tf.size(err), err.dtype)
        self.total_err.assign_add(tf.reduce_sum(err))
        self.total_w.assign_add(w)

    def result(self):
        return self.total_err / (self.total_w + 1e-8)

    def reset_state(self):
        self.total_err.assign(0.0)
        self.total_w.assign(0.0)


In [11]:
# === Cellule 10 : Squeezeformer-lite block ===
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

class ConvModule(layers.Layer):
    """
    Squeezeformer-style conv module:
      - PW Conv 1x1 -> 2*d (pour GLU)
      - GLU (a * sigmoid(b))
      - Depthwise Conv (k)
      - BN + SiLU
      - PW Conv 1x1 -> d
      - Dropout
    Implémenté en pur Keras (aucun tf.* au build-time).
    """
    def __init__(self, d_model, kernel=15, dropout=0.1, name=None):
        super().__init__(name=name)
        self.d = d_model
        self.pw_in   = layers.Conv1D(2*d_model, kernel_size=1, padding="same")
        # depthwise 1D portable : SeparableConv1D (depthwise + pointwise)
        # on met pointwise filters = d_model (classique “sepconv”)
        self.dw_sep  = layers.SeparableConv1D(d_model, kernel_size=kernel, padding="same")
        self.bn      = layers.BatchNormalization()
        self.act     = layers.Activation("swish")  # SiLU
        self.pw_out  = layers.Conv1D(d_model, kernel_size=1, padding="same")
        self.drop    = layers.Dropout(dropout)

    def call(self, x, training=False):
        # x: (B, L, d)
        x = self.pw_in(x)                       # (B, L, 2d)
        # GLU sans tf.split “à la construction” : slice dans call
        a = x[..., :self.d]
        b = x[..., self.d:]
        x = a * tf.sigmoid(b)                   # (B, L, d)
        x = self.dw_sep(x)                      # (B, L, d)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.pw_out(x)                      # (B, L, d)
        return self.drop(x, training=training)

class PointwiseFFN(layers.Layer):
    def __init__(self, d_model, mult=4, dropout=0.1, name=None):
        super().__init__(name=name)
        self.fc1  = layers.Dense(d_model*mult, activation="gelu")
        self.do1  = layers.Dropout(dropout)
        self.fc2  = layers.Dense(d_model)
        self.do2  = layers.Dropout(dropout)
        self.ln   = layers.LayerNormalization()

    def call(self, x, training=False):
        h = self.fc1(x)
        h = self.do1(h, training=training)
        h = self.fc2(h)
        h = self.do2(h, training=training)
        return self.ln(x + h)   # résiduel + LN

class SqueezeBlock(layers.Layer):
    """
    (FFN -> MHSA -> ConvModule -> FFN) avec résidu/LN entre étapes.
    `attn_mask`: bool (B, L, L).
    """
    def __init__(self, d_model, n_heads, ffn_mult=4, dropout=0.1, conv_kernel=15, name=None):
        super().__init__(name=name)
        self.ffn1  = PointwiseFFN(d_model, mult=ffn_mult, dropout=dropout, name="ffn1")
        self.mha   = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model//n_heads, dropout=dropout)
        self.do_mh = layers.Dropout(dropout)
        self.ln_mh = layers.LayerNormalization()
        self.conv  = ConvModule(d_model, kernel=conv_kernel, dropout=dropout, name="conv")
        self.ln_cv = layers.LayerNormalization()
        self.ffn2  = PointwiseFFN(d_model, mult=ffn_mult, dropout=dropout, name="ffn2")

    def call(self, x, attn_mask=None, training=False):
        # 1) FFN pré-attention
        x = self.ffn1(x, training=training)

        # 2) MHA + résiduel
        h = self.mha(query=x, value=x, key=x, attention_mask=attn_mask, use_causal_mask=False, training=training)
        h = self.do_mh(h, training=training)
        x = self.ln_mh(x + h)

        # 3) Conv module + résiduel
        h = self.conv(x, training=training)
        x = self.ln_cv(x + h)

        # 4) FFN post
        x = self.ffn2(x, training=training)
        return x


In [12]:
# === Cellule 11 : Modèle Squeezeformer + BPPM (fix mask) ===

# Hypers (identiques)
VOCAB_SIZE = 5
EMB_DIM    = 128
D_MODEL    = 192
N_HEADS    = 4
N_BLOCKS   = 4
DROPOUT    = 0.1
CONV_K     = 15

# Inputs alignés
seq_in  = keras.Input(shape=(MAX_LEN,), dtype="int32",   name="seq")
bppm_in = keras.Input(shape=(MAX_LEN, BPPM_band.shape[-1]), dtype="float32", name="bppm")

# Embedding + proj
tok = layers.Embedding(VOCAB_SIZE, EMB_DIM, mask_zero=False, name="tok_emb")(seq_in)
h   = layers.Dense(D_MODEL, name="proj_dmodel")(tok)

# Masques
valid_mask = layers.Lambda(
    lambda x: tf.not_equal(x, PAD_ID),
    output_shape=(MAX_LEN,),
    name="valid_mask"
)(seq_in)  # (B,L,L)
attn_mask = layers.Lambda(
    lambda m: tf.cast(m[:, :, None] & m[:, None, :], tf.bool),
    output_shape=(MAX_LEN, MAX_LEN),
    name="attn_mask"
)(valid_mask)
# Blocs
for i in range(N_BLOCKS):
    h = SqueezeBlock(d_model=D_MODEL, n_heads=N_HEADS, ffn_mult=4,
                     dropout=DROPOUT, conv_kernel=CONV_K, name=f"squeeze_{i}")(h, attn_mask=attn_mask)

# BPPM -> petite proj + fusion
b = layers.Dense(64, activation="relu", name="bppm_proj")(bppm_in)
h = layers.Concatenate(axis=-1, name="fuse_seq_bpp")([h, b])
h = layers.Dense(128, activation="gelu", name="post_fuse_dense")(h)
h = layers.Dropout(DROPOUT, name="post_fuse_dropout")(h)

out = layers.Dense(2, activation="linear", name="reactivity")(h)

model = keras.Model(inputs={"seq": seq_in, "bppm": bppm_in}, outputs=out, name="squeezeformer_bppm")
model.summary()


Model: "squeezeformer_bppm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq (InputLayer)    │ (None, 206)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tok_emb (Embedding) │ (None, 206, 128)  │        640 │ seq[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ valid_mask (Lambda) │ (None, 206)       │          0 │ seq[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ proj_dmodel (Dense) │ (None, 206, 192)  │     24,768 │ tok_emb[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_mask (Lambda)  │ (None, 206, 206)  │          0 │ valid_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ squeeze_0           │ (None, 206, 192)  │    893,376 │ proj_dmodel[0][0… │
│ (SqueezeBlock)      │                   │            │ attn_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ squeeze_1           │ (None, 206, 192)  │    893,376 │ squeeze_0[0][0],  │
│ (SqueezeBlock)      │                   │            │ attn_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ squeeze_2           │ (None, 206, 192)  │    893,376 │ squeeze_1[0][0],  │
│ (SqueezeBlock)      │                   │            │ attn_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bppm (InputLayer)   │ (None, 206, 41)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ squeeze_3           │ (None, 206, 192)  │    893,376 │ squeeze_2[0][0],  │
│ (SqueezeBlock)      │                   │            │ attn_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bppm_proj (Dense)   │ (None, 206, 64)   │      2,688 │ bppm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fuse_seq_bpp        │ (None, 206, 256)  │          0 │ squeeze_3[0][0],  │
│ (Concatenate)       │                   │            │ bppm_proj[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ post_fuse_dense     │ (None, 206, 128)  │     32,896 │ fuse_seq_bpp[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ post_fuse_dropout   │ (None, 206, 128)  │          0 │ post_fuse_dense[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reactivity (Dense)  │ (None, 206, 2)    │        258 │ post_fuse_dropou… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,634,754 (13.87 MB)

 Trainable params: 3,633,218 (13.86 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [13]:
# === Cellule 12 : Compile + callbacks (fix) ===
ckpt_path = str(CACHE_DIR / "squeezeformer_best.keras")

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss=mae_clipped_elementwise,               # élémentaire, pondérée par sample_weight
    metrics=[MaskedMAE(name="masked_mae")],     # nom explicite
    run_eagerly=False
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        ckpt_path,
        monitor="val_masked_mae",
        mode="min",                # ⬅️ important
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_masked_mae",
        mode="min",                # ⬅️ important
        patience=3,
        factor=0.5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_masked_mae",
        mode="min",                # ⬅️ important
        patience=7,
        restore_best_weights=True,
        verbose=1
    )
]


In [ ]:
# === Cellule 13 : Entraînement ===

history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=20,
    #callbacks=callbacks,
    verbose=1
)


Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 57s 6s/step - loss: 0.1415 - masked_mae: 0.2428 - val_loss: 0.1328 - val_masked_mae: 0.2267
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 42s 6s/step - loss: 0.1271 - masked_mae: 0.2084 - val_loss: 0.1287 - val_masked_mae: 0.2194
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 42s 6s/step - loss: 0.1250 - masked_mae: 0.2073 - val_loss: 0.1304 - val_masked_mae: 0.2330
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 42s 6s/step - loss: 0.1218 - masked_mae: 0.2081 - val_loss: 0.1236 - val_masked_mae: 0.2622
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 83s 6s/step - loss: 0.1133 - masked_mae: 0.2028 - val_loss: 0.1111 - val_masked_mae: 0.2154
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 82s 6s/step - loss: 0.1063 - masked_mae: 0.1984 - val_loss: 0.1068 - val_masked_mae: 0.2269
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 42s 6s/step - loss: 0.1033 - masked_mae: 0.2003 - val_loss: 0.1080 - val_masked_mae: 0.2172
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 42s 6s/step - loss: 0.1018 - masked_mae: 0.2007 - val_loss: 0.1015 -

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_mae = history.history.get("masked_mae", [])
val_mae   = history.history.get("val_masked_mae", [])

# --- Ajout du point à l'époque 0 uniquement pour le train ---
train_mae = [0.22] + train_mae

def smooth_curve(values, window=3):
    if len(values) < window or window <= 1:
        return np.array(values, dtype=float)
    kernel = np.ones(window) / window
    y = np.convolve(values, kernel, mode='same')
    # conserve exactement les valeurs aux bords
    y[0]  = values[0]
    y[-1] = values[-1]
    return y

window = 3
smooth_train = smooth_curve(train_mae, window)
smooth_val   = smooth_curve(val_mae, window)

epochs_train = np.arange(len(train_mae))  # 0,1,2,...
epochs_val   = np.arange(len(val_mae))    # 0,1,2,...

plt.figure(figsize=(8,6))
plt.plot(epochs_train, smooth_train, label="Train MAE")
plt.plot(epochs_val, smooth_val,   label="Val MAE")
plt.xlabel("Époch")
plt.ylabel("MAE")
plt.title("MAE")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# === Cellule 14 : Évaluation & sanity check ===
val_metrics = model.evaluate(ds_valid, verbose=0)
print({"val_loss": float(val_metrics[0]), "val_mae_clipped": float(val_metrics[1])})

# Sanity: une poignée d'exemples
bx, by, bw = next(iter(ds_valid))
yp = tf.clip_by_value(model.predict(bx, verbose=0), 0.0, 1.0)

# Moyenne par canal (DMS/A3) sur positions valides
w2 = bw.numpy()  # (B,L,2)
mask = w2 > 0
mae_per_sample = np.sum(np.abs(by.numpy() - yp) * mask, axis=(1,2)) / np.sum(mask, axis=(1,2)).clip(min=1)
print("MAE@valid (5 premiers):", np.round(mae_per_sample[:5], 4))


In [ ]:
# Récupération des historiques de perte
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

# Affichage formaté des valeurs
print("Validation Loss:", [f"{v:.8f}" for v in val_loss])
print("Training Loss:", [f"{v:.8f}" for v in train_loss])

# Visualisation des courbes d'entraînement
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(train_loss, label="Train loss", markersize=3)
plt.plot(val_loss, label="Val loss", markersize=3)

# Définition des bornes avec une petite marge
lo, hi = min(val_loss + train_loss), max(val_loss + train_loss)
pad = max(1e-5, 0.2 * (hi - lo))
plt.ylim(lo - pad, hi + pad)

plt.xlabel("Époque")
plt.ylabel("Loss")
plt.title("Évolution des pertes d'entraînement et de validation")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# même masquage que ds_valid: sample_weight = W_pos répété sur 2 canaux
val_loss_manual = []
for (bx, by, bw) in ds_valid:
    w2 = bw.numpy()
    yp = model.predict(bx, verbose=0)
    yp = np.clip(yp, 0.0, 1.0)
    num = np.sum(np.abs(by.numpy() - yp) * w2)
    den = np.sum(w2) + 1e-12
    val_loss_manual.append(num/den)
print("val_loss(manual) =", float(np.mean(val_loss_manual)))


In [ ]:
# MAE global + par canal + par longueur (fix)
import numpy as np
import pandas as pd
from collections import defaultdict

def eval_ds(ds, model):
    maes = defaultdict(list)

    for bx, by, bw in ds:
        # Prédiction
        yp = model.predict(bx, verbose=0)
        yp = np.clip(yp, 0.0, 1.0)

        # Masque positions valides
        mask = bw.numpy() > 0  # (B,L,2)
        abs_err = np.abs(by.numpy() - yp) * mask

        # Global
        maes["global"].append(abs_err.sum((1,2)) / mask.sum((1,2)).clip(min=1))

        # Par canal
        for ci, name in enumerate(["DMS", "2A3"]):
            m = mask[:, :, ci]; e = abs_err[:, :, ci]
            maes[name].append(e.sum(1) / m.sum(1).clip(min=1))

        # Par longueur réelle (compte les PAD=0)
        lens = (bx["seq"].numpy() != PAD_ID).sum(1)              # (B,)
        bins = pd.Series(pd.cut(lens, bins=[0,170,190,1000],
                                labels=["<=170","171-190",">190"]))  # Series catégorielle

        for lbl in bins.cat.categories:          # <=170 / 171-190 / >190
            sel = (bins == lbl).to_numpy()       # (B,) bool
            if sel.any():
                maes[f"len:{lbl}"].append(
                    abs_err[sel].sum((1,2)) / mask[sel].sum((1,2)).clip(min=1)
                )

    # Moyenne sur tous les batches
    out = {k: float(np.mean(np.concatenate(v))) for k, v in maes.items()}
    return out

metrics_table = eval_ds(ds_valid, model)
print(pd.Series(metrics_table).sort_index())


In [ ]:
# === Cellule B1 (FAST) : Prépare le jeu de test (mêmes steps que train) ===
assert TEST_CSV.exists(), f"Introuvable: {TEST_CSV}"

# 0) Param de debug pour accélérer (None = tout le test)
SUBTEST_N = 20000             # ex. mets 500 ou 2000 pour tester vite
BATCH_ENCODE = 8192          # taille de chunk pour l'encodage vectorisé

# 1) Lecture CSV minimale
test_df = pd.read_csv(TEST_CSV, usecols=["sequence_id", "sequence"], dtype={"sequence_id": str})
if SUBTEST_N is not None:
    test_df = test_df.iloc[:SUBTEST_N].copy()

seq_ids_test = test_df["sequence_id"].to_numpy()
seq_str_test = test_df["sequence"].astype(str).to_numpy()

# 2) Encodage/padding rapide (vectorisé par chunks pour éviter un énorme Python loop)
def encode_chunk(arr):
    return np.stack([encode_and_pad(s) for s in arr], axis=0)

X_seq_test_list = []
for i in range(0, len(seq_str_test), BATCH_ENCODE):
    X_seq_test_list.append(encode_chunk(seq_str_test[i:i+BATCH_ENCODE]))
X_seq_test = np.concatenate(X_seq_test_list, axis=0).astype("int32")

# 3) BPP: RÉUTILISER l’index si déjà créé en train (évite le rescan disque)
if "bpp_paths" not in globals() or not isinstance(bpp_paths, dict) or len(bpp_paths) == 0:
    print("[BPP] index absent en mémoire → indexation minimale…")
    # ⚠️ Si tu dois réindexer: fais-le UNE fois puis persiste-le (voir plus bas).
    bpp_paths = index_bpp_folder(BPP_ROOT)
else:
    print(f"[BPP] index réutilisé (len={len(bpp_paths)})")

# 4) Chargement BPP en parallèle (threads I/O-bound)
from multiprocessing.pool import ThreadPool

def band_for_id(sid: str):
    p = bpp_paths.get(str(sid))
    if p is None:
        return np.zeros((MAX_LEN, LOCAL_W), dtype=np.float32)
    return load_band_from_txt_fast(p, band=BAND, Lmax=MAX_LEN)

N_WORKERS = 12  # ajuste selon la machine (8–16)
with ThreadPool(processes=N_WORKERS) as pool:
    bands = list(pool.imap(band_for_id, seq_ids_test, chunksize=32))

BPPM_band_test = np.stack(bands, axis=0).astype("float32")

print("X_seq_test:", X_seq_test.shape, "| BPPM_band_test:", BPPM_band_test.shape)


In [ ]:
# === Cellule B2 : Inférence test (modèle en RAM) ===
import numpy as np
import tensorflow as tf

# Juste au cas où (cohérence des types)
X_seq_test = X_seq_test.astype("int32")
BPPM_band_test = BPPM_band_test.astype("float32")

BATCH_TEST = 1024  # adapte si OOM

y_pred_test = model.predict(
    {"seq": X_seq_test, "bppm": BPPM_band_test},
    batch_size=BATCH_TEST,
    verbose=1
)
y_pred_test = np.clip(y_pred_test, 0.0, 1.0).astype("float32")

print("y_pred_test:", y_pred_test.shape, y_pred_test.dtype,
      "| min/max:", float(y_pred_test.min()), float(y_pred_test.max()))


In [ ]:
# === Cellule B3 : Construire le DataFrame de sortie & aligner avec sample_submission ===
pred_cols = [f"reactivity_{k:04d}" for k in range(1, MAX_LEN+1)]

# Choix simple : moyenne des 2 canaux (ou garder 2 canaux si requis)
# Ici on garde les 2 canaux séparés, suffixés _DMS et _2A3, *et* on fournit aussi une moyenne si besoin.
pred_wide = pd.DataFrame({"sequence_id": seq_ids_test})
pred_dms  = pd.DataFrame(y_pred_test[:, :, 0], columns=[c+"_DMS" for c in pred_cols])
pred_2a3  = pd.DataFrame(y_pred_test[:, :, 1], columns=[c+"_2A3" for c in pred_cols])
pred_mean = pd.DataFrame(y_pred_test.mean(axis=2), columns=[c for c in pred_cols])

pred_df = pd.concat([pred_wide, pred_dms, pred_2a3, pred_mean], axis=1)

# Si un sample_submission existe, on s’aligne sur ses colonnes
sub_path = DATA_DIR / "sample_submission.csv"
if sub_path.exists():
    sub = pd.read_csv(sub_path)
    cols = [c for c in sub.columns if c in pred_df.columns]
    # On merge par sequence_id si présent, sinon on remplit colonnes direct si même ordre
    if "sequence_id" in sub.columns and "sequence_id" in pred_df.columns:
        out = sub[["sequence_id"]].merge(pred_df, on="sequence_id", how="left")
        # Conserve exactement les colonnes de sub quand possible
        final_cols = sub.columns
        for c in sub.columns:
            if c in pred_df.columns:
                out[c] = pred_df[c]
        out = out[final_cols]
    else:
        out = sub.copy()
        for c in cols:
            out[c] = pred_df[c].values
else:
    # Pas de sample: on sauve notre format “riche”
    out = pred_df

OUT_PRED = CACHE_DIR / "predictions2_test.csv"
out.to_csv(OUT_PRED, index=False)
print("Écrit:", OUT_PRED)
print("Colonnes de sortie:", len(out.columns))


In [ ]:
import csv

with open("cache/predictions_test.csv", newline="") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        print(row)
        if i == 4:  # affiche les 5 premières lignes
            break


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# y_pred_test: (N, L, 2) après clip dans [0,1]
y_pred_test = np.clip(y_pred_test, 0, 1)

sub_path = Path(DATA_DIR) / "sample_submission.csv"
sub = pd.read_csv(sub_path)

# Aplatir dans le même ordre que tu as passé les séquences au modèle
dms  = y_pred_test[:, :, 0].reshape(-1)
a2a3 = y_pred_test[:, :, 1].reshape(-1)

n = len(sub)
sub["reactivity_DMS_MaP"]  = dms[:n]
sub["reactivity_2A3_MaP"]  = a2a3[:n]

OUT = Path(CACHE_DIR) / "predictions_test.csv"
sub.to_csv(OUT, index=False)
print("Écrit :", OUT)
print(sub.head())
